In [1]:
import os,sys
 

os.environ["PYSPARK_DRIVER_PYTHON"] = sys.executable
os.environ["PYSPARK_PYTHON"]        = sys.executable

In [2]:
from pyspark import SparkContext
from sympy import *
from drudge import *
import time
#from gristmill import *
print("finished importing")

finished importing


In [3]:
import re

# ----------------------------------------------------------------------------
# 1) Start Spark & BCS quasiparticle drudge
# ----------------------------------------------------------------------------
ctx = SparkContext('local[*]', 'bcs_ccsd')
dr  = ReducedBCSDrudge(ctx)
dr.full_simplify = True
print("finished")

26/03/10 11:29:36 WARN Utils: Your hostname, Swarnamoys-MacBook-Pro.local resolves to a loopback address: 127.0.0.1; using 168.5.62.164 instead (on interface en0)
26/03/10 11:29:36 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/03/10 11:29:36 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
                                                                                

finished


In [4]:
P, Pdag, N = dr.names.P, dr.names.Pdag, dr.names.N
p,q,r,s,i,j,k,l = dr.names.A_dumms[:8]
u, v = IndexedBase('u'), IndexedBase('v')

P_i_dag = (u[p]*v[p]+u[p]**2*Pdag[p]
     - u[p]*v[p]*N[p]
     - v[p]**2*P[p])

P_j_dag = (
        u[q]*v[q]
      + u[q]**2 * Pdag[q]
      - u[q]*v[q] * N[q]
      - v[q]**2 * P[q]
    )

P_k_dag = (
        u[r]*v[r]
      + u[r]**2 * Pdag[r]
      - u[r]*v[r] * N[r]
      - v[r]**2 * P[r]
    )
P_i = (
        (u[p])*(v[p])
      + (u[p])**2 * P[p]
      - (u[p])*(v[p]) * N[p]
      - (v[p])**2 * Pdag[p]
    )
P_j = ((u[q])*(v[q])
      + (u[q])**2 * P[q]
      - (u[q])*(v[q]) * N[q]
      - (v[q])**2 * Pdag[q])

P_k = ((u[r])*(v[r])
      + (u[r])**2 * P[r]
      - (u[r])*(v[r]) * N[r]
      - (v[r])**2 * Pdag[r])
#hc = P_j_dag * P_i
N_i = 2*(v[p])*v[p] +((u[p])*u[p] - (v[p])*v[p])*N[p] +2*u[p]*(v[p])*Pdag[p]\
            +2*(u[p])*v[p]*P[p]
N_j = 2*(v[q])*v[q] +((u[q])*u[q] - (v[q])*v[q])*N[q] +2*u[q]*(v[q])*Pdag[q]\
            +2*(u[q])*v[q]*P[q]

N_k = 2*(v[r])*v[r] +((u[r])*u[r] - (v[r])*v[r])*N[r] +2*u[r]*(v[r])*Pdag[r]\
       +2*(u[r])*v[r]*P[r]


In [5]:
H11, H20 , H02 = IndexedBase('H11'), IndexedBase('H20'), IndexedBase('H02')
H04, H40 = IndexedBase('H04'), IndexedBase('H40')
H22, Hb22 = IndexedBase('H22'), IndexedBase('HT22')
H31, H13 = IndexedBase('H31'), IndexedBase('H13')
W,V = IndexedBase('W'), IndexedBase('V')
h = IndexedBase('h')
dr.set_symm(H04,Perm([1,0],IDENT))
dr.set_symm(H40,Perm([1,0],IDENT)) 

dr.set_symm(W, Perm([1, 0]), valence=2)
dr.set_symm(V, Perm([1, 0]), valence=2)


In [6]:
expr = h[p]*N[p] + Rational(1,4)*W[p,q]*N[p]*N[q] + V[p,q]*Pdag[p]*P[q]
expr = dr.einst(expr).simplify()
expr.display()

<IPython.core.display.Math object>

In [7]:
expr = h[p]*N_i + Rational(1,4)*W[p,q]*N_i*N_j + V[p,q]*P_i_dag*P_j
expr = dr.einst(expr).simplify().merge()
expr.display()

<IPython.core.display.Math object>

In [8]:
t1 = IndexedBase('t1')
t2 = IndexedBase('t2')
t3 = IndexedBase('t3')
t4 = IndexedBase('t4')
dr.set_symm(t2, Perm([1, 0],IDENT), valence=2)
dr.set_symm(t3, Perm([1,0,2],IDENT), Perm([0,2,1],IDENT), valence=3) 

T1 = dr.einst(t1[p] * Pdag[p])
T2 = dr.einst(t2[p,q] *Pdag[p]*Pdag[q])/2
T3 = dr.einst(Rational(1,6)*t3[p,q,r]*Pdag[p]*Pdag[q]*Pdag[r])
T4 = dr.einst(t4[p,q,r,s]*Pdag[p]*Pdag[q]*Pdag[r]*Pdag[s])/24


cluster =  T1+T2#+T3#+T4
cluster.display()

<IPython.core.display.Math object>

In [9]:
t0 = time.time()
Hbar = expr
curr = expr
fact = 1
for n in range(5):                              # up to 4-fold for 2-body H
    comm = (curr | cluster).simplify()          # ad_T^{n+1}(H_N)
    fact *= (n + 1)                             # 1!,2!,3!,4!
    Hbar += comm / fact                         # add 1/(n+1)! term
    curr = comm    
    print(n)
print("Hbar evaluation done ", time.time()-t0)

0


1


2


3


[Stage 162:================================================>   (270 + 12) / 288]

4
Hbar evaluation done  21.677261114120483


In [10]:
from drudge import Term

def coeff_with_internal_sums(ts):
    """
    Strip the operator word and *only* those sums that run over
    indices appearing in the operator word. Keep any other internal sums. We need this for Integrals.
    """
    def proc(t):
        # Collect indices used in the operator word
        op_inds = set()
        for v in t.vecs:
            # depending on your Drudge version, this may be v.indices or v.inds
            for ind in v.indices:
                op_inds.add(ind)

        # Keep only sums whose index is NOT in the operator word
        new_sums = tuple((i, base) for (i, base) in t.sums if i not in op_inds)

        # Strip vecs; keep amp unchanged
        return Term(new_sums, t.amp, ())

    return ts.map(proc)

def coeff_only(ts):
    """Return a TermSum with the same sums/amp but with vecs removed."""
    # ts can be a TermSum or anything with .map over terms
    return ts.map(lambda t: Term(t.sums, t.amp, ()))   # vecs = ()

In [11]:
# Loop through each local term in H_N
for i, term in enumerate(expr.local_terms):
    print(f"--- Term {i} ---")
       # The scalar coefficient
    print("Operators (vecs):", term.vecs)   # The associated vector operators
    print()

--- Term 0 ---
Operators (vecs): (Vec('N', (p)),)

--- Term 1 ---
Operators (vecs): (Vec('P^\\dagger', (p)), Vec('P^\\dagger', (q)))

--- Term 2 ---
Operators (vecs): ()

--- Term 3 ---
Operators (vecs): (Vec('P^\\dagger', (p)), Vec('P', (q)))

--- Term 4 ---
Operators (vecs): (Vec('P', (p)),)

--- Term 5 ---
Operators (vecs): ()

--- Term 6 ---
Operators (vecs): (Vec('P^\\dagger', (p)),)

--- Term 7 ---
Operators (vecs): (Vec('P', (p)),)

--- Term 8 ---
Operators (vecs): (Vec('P', (p)), Vec('P', (q)))

--- Term 9 ---
Operators (vecs): (Vec('N', (p)),)

--- Term 10 ---
Operators (vecs): (Vec('P^\\dagger', (p)), Vec('N', (q)))

--- Term 11 ---
Operators (vecs): (Vec('N', (p)), Vec('P', (q)))

--- Term 12 ---
Operators (vecs): (Vec('N', (p)), Vec('N', (q)))

--- Term 13 ---
Operators (vecs): (Vec('P^\\dagger', (p)),)



In [12]:
def get_Pdag(term):
    vecs = term.vecs
    # Must be exactly 1 operators
    if len(vecs) != 1:
        return False
    labels = [v.label for v in vecs]
     
    return labels == ['P^\\dagger']

def get_P(term):
    vecs = term.vecs
    # Must be exactly 1 operators
    if len(vecs) != 1:
        return False
    labels = [v.label for v in vecs]
     
    return labels == ['P']

def get_N(term):
    vecs = term.vecs
    # Must be exactly 1 operators
    if len(vecs) != 1:
        return False
    labels = [v.label for v in vecs]
     
    return labels == ['N']


def get_NN(term):
    vecs = term.vecs
    # Must be exactly 1 operators
    if len(vecs) != 2:
        return False
    labels = [v.label for v in vecs]
     
    return labels == ['N','N']

def get_NP(term):
    vecs = term.vecs
    # Must be exactly 1 operators
    if len(vecs) != 2:
        return False
    labels = [v.label for v in vecs]
     
    return labels == ['N','P']

def get_PP(term):
    vecs = term.vecs
    # Must be exactly 1 operators
    if len(vecs) != 2:
        return False
    labels = [v.label for v in vecs]
     
    return labels == ['P','P']

def get_PdagPdag(term):
    vecs = term.vecs
    # Must be exactly 1 operators
    if len(vecs) != 2:
        return False
    labels = [v.label for v in vecs]
     
    return labels == ['P^\\dagger','P^\\dagger']

def get_PdagN(term):
    vecs = term.vecs
    # Must be exactly 1 operators
    if len(vecs) != 2:
        return False
    labels = [v.label for v in vecs]
     
    return labels == ['P^\\dagger','N']

def get_PdagP(term):
    vecs = term.vecs
    # Must be exactly 1 operators
    if len(vecs) != 2:
        return False
    labels = [v.label for v in vecs]
     
    return labels == ['P^\\dagger','P']


In [13]:
tensor0_ = expr.filter(lambda term: len(term.vecs) == 0)
#tensor0_ = dr.einst(tensor0_).simplify().merge().merge()
tensor0_ = tensor0_.simplify().merge().merge()
tensor0_.display() 
#print(tensor0_.latex())

<IPython.core.display.Math object>

In [14]:
tensor_ = expr.filter(lambda term: len(term.vecs) == 2)
tensor_ = dr.einst(tensor_).simplify().merge().merge()
tensor_.display()


<IPython.core.display.Math object>

In [16]:
pp_terms = tensor_.terms.filter(get_PP)
H40_dr = Tensor(dr, coeff_with_internal_sums(pp_terms)).simplify().merge()
#print(H40_dr.latex())
H40_dr.display()
 

<IPython.core.display.Math object>